# EDA — Simulation Sales Data

Exploratory analysis of 18 months of synthetic operational data from a simulated dog daycare (Jan 2023 – Jun 2024).  
Data loaded from the SQLite warehouse populated by `scripts/run_pipeline.py`.

In [ ]:
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.dpi"] = 110

DB_PATH = Path("../data/simulation_dev.db")
conn = sqlite3.connect(DB_PATH)

sales = pd.read_sql("SELECT * FROM sales", conn, parse_dates=["date"])
customers = pd.read_sql("SELECT * FROM customer_features", conn)

print(f"Sales rows : {len(sales):,}")
print(f"Customers  : {len(customers)}")
sales.head(3)

## Revenue by month

In [ ]:
monthly = (
    sales.assign(month=sales["date"].dt.to_period("M"))
    .groupby("month")["price_mxn"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "revenue", "count": "transactions"})
    .reset_index()
)
monthly["month_dt"] = monthly["month"].dt.to_timestamp()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax2 = ax1.twinx()

ax1.bar(monthly["month_dt"], monthly["revenue"] / 1000, width=20,
        color="steelblue", alpha=0.7, label="Revenue (k MXN)")
ax2.plot(monthly["month_dt"], monthly["transactions"],
         color="tomato", marker="o", linewidth=1.5, label="Transactions")

ax1.set_ylabel("Revenue (thousands MXN)")
ax2.set_ylabel("Transaction count")
ax1.set_xlabel("Month")
fig.suptitle("Monthly Revenue and Transaction Volume", fontsize=13)
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.9))
plt.tight_layout()
plt.show()

## Service mix

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

svc_vol = sales["service"].value_counts()
svc_vol.plot.bar(ax=axes[0], color=sns.color_palette("muted", len(svc_vol)))
axes[0].set_title("Transaction volume by service")
axes[0].set_ylabel("Transactions")
axes[0].tick_params(axis="x", rotation=30)

svc_rev = sales.groupby("service")["price_mxn"].sum().sort_values(ascending=False)
svc_rev.div(1000).plot.bar(ax=axes[1], color=sns.color_palette("muted", len(svc_rev)))
axes[1].set_title("Revenue by service (k MXN)")
axes[1].set_ylabel("Revenue (thousands MXN)")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

## Payment method distribution

In [ ]:
pay = sales["payment_method"].value_counts()
fig, ax = plt.subplots(figsize=(5, 4))
ax.pie(pay.values, labels=pay.index, autopct="%1.1f%%",
       colors=sns.color_palette("pastel", len(pay)), startangle=90)
ax.set_title("Payment method split")
plt.tight_layout()
plt.show()

## Churn segments

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label in zip(
    axes,
    ["days_since_last_visit", "visit_frequency", "avg_spend"],
    ["Days since last visit", "Visit frequency (visits/month)", "Avg spend (MXN)"],
):
    for churn_val, grp_label, color in [(0, "Active", "steelblue"), (1, "Churned", "tomato")]:
        subset = customers[customers["is_churned"] == churn_val][col]
        ax.hist(subset, bins=25, alpha=0.6, label=grp_label, color=color, density=True)
    ax.set_title(label)
    ax.legend()

fig.suptitle("Feature distributions: Active vs Churned customers", fontsize=13)
plt.tight_layout()
plt.show()

print(customers.groupby("is_churned")[["days_since_last_visit", "visit_frequency", "avg_spend"]].mean().round(1))

## Top 10 customers by revenue

In [ ]:
top10 = customers.nlargest(10, "total_revenue")[["customer_id", "total_revenue", "visit_frequency", "is_churned"]]
top10["is_churned"] = top10["is_churned"].map({0: "Active", 1: "Churned"})
top10.style.format({"total_revenue": "${:,.0f} MXN", "visit_frequency": "{:.1f}"})